In [ ]:
# Multi Agent System

In [ ]:
# 1 . Langchain
# 2. Langgraph
# 3. Open AI Agent Sdk
# 4. n8n
# 5. crew ai

# Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

In [ ]:
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

In [ ]:
# # openai agent sdk -> control , handsoff ,gaurddrails (openai) -> Travel agent
# crewai -> Task ->crew  (control level )

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [ ]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's model.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [ ]:
import os
from crewai import Agent
from langchain_openai import ChatOpenAI

# Configure the OpenAI LLM
openai_llm = ChatOpenAI(
    api_key="your_api_key_here",
    base_url="https://models.github.ai/inference",
    model="gpt-4o-mini"
)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:937: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `CrewAgentExecutor` to V2.
  warnings.warn(


In [ ]:
openai_llm.invoke("Hello")

AIMessage(content='Hello! How can I assist you today?', response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 8, 'total_tokens': 18, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini', 'system_fingerprint': 'fp_f97eff32c5', 'finish_reason': 'stop', 'logprobs': None}, id='run-3b5460cf-3ce3-4fba-9fe4-cffd77d17438-0')

In [ ]:
# 1. system message ( ROle)
# 2. Human Message
# 3. AIMessage

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

In [ ]:
# Agent :

# 1. role :
# 2. goal : ......
# 3. Backstory : "Youre are an Expert Solution Architect from Microsoft with 20+ years of experience,"


In [ ]:
# prompt ="""
# ....................
# .....................
# ""


# ".......... "
# ".........."

In [ ]:
# @ agent  done
# @ task done
# @ crew done

# starting -> input -> crew

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [ ]:
# Crew AI : Framework for Multi-Agent Systems

In [ ]:
# topic -> Article Generate  | planner agent (topic) , writter (topic) , editor (topic)

In [ ]:
# Topic -> Article Generate

# 1. Planner agent -> Full content planner  {}
# 2. Writter agent -> Full write content based on the content got from planner agent
# 3. Editor agent -> Fix


In [ ]:
# Planner Agent -> Writter Agent -> Editor Agent

In [ ]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
  allow_delegation=False,
	verbose=True,
  llm=openai_llm,

)

### Agent: Writer

In [ ]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True,
    llm=openai_llm
)

### Agent: Editor

In [ ]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True,
    llm=openai_llm
)

In [ ]:
# agent  task , crew

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [ ]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

In [ ]:
# Task[.....] | Task [...] |


# Main agent -> Task -> Agent

### Task: Write

In [ ]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [ ]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor,
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution.

In [ ]:
# 3 Task , 3 agenerate_from_stream
# crew (task ,agent,verbose)

In [ ]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [ ]:
result = crew.kickoff(inputs={"topic": "Winter in Bangladesh"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Winter in Bangladesh.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer.  
Final Answer:

### Content Plan for Blog Article: "Winter in Bangladesh"

---

#### 1. Latest Trends, Key Players, and Noteworthy News

- **Trends:**
  - Increasing popularity of winter tourism in Bangladesh, especially in hill tracts and rural areas.
  - Emergence of winter festivals showcasing local culture, crafts, and cuisines.
  - Growing interest in sustainable travel options during the winter months.

- **Key Players:**
  - Local tourism boards promoting winter destinations.
  - NGOs focusing on rural development thro

In [ ]:
result

"```markdown\n# Winter in Bangladesh: A Season of Culture and Adventure\n\n## I. Introduction\n\nWinter in Bangladesh, spanning from November to February, is a cherished season that brings a delightful change to the landscape and lifestyle of its people. As the temperatures dip, ranging from a refreshing 10°C to a comfortable 20°C, the country transforms into a canvas of vibrant festivities, cultural richness, and stunning natural beauty. This season not only marks a time for celebration but also invites locals and tourists alike to engage deeply with the country's traditions and outdoor adventures.\n\nThe mild winter weather offers a reprieve from the intense heat and humidity that characterize other seasons in Bangladesh. It is a time when families gather, friends reunite, and communities come alive with events that celebrate the harvest and the spirit of togetherness. Whether you are a local resident or a visitor, winter in Bangladesh is an experience that captivates the senses and 

- Display the results of your execution as markdown in the notebook.

In [ ]:
from IPython.display import Markdown
Markdown(result)

```markdown
# Understanding Sustainable Energy: Trends, Technologies, and Smart Choices for a Greener Future  

## Introduction  

Sustainable energy represents a transformative solution to combating climate change and addressing the global energy crisis. With the world grappling with rising greenhouse gas emissions and dwindling fossil fuel reserves, transitioning to cleaner energy sources has never been more critical. Renewable energy, in particular, offers a path to a greener future, generating 29% of global electricity in 2022—an impressive leap compared to previous decades.  

In this blog post, we’ll explore the latest trends, advancements, and innovations in sustainable energy, as well as practical ways for individuals, businesses, and governments to make informed decisions about adopting renewable energy solutions. By the end, you’ll have a clearer understanding of how sustainable energy can benefit both the planet and your pocket.  

---

## The Rise of Renewable Energy  

Renewable energy has emerged as a cornerstone of sustainable development, offering environmentally friendly alternatives to fossil fuels. Solar power, harnessing the sun’s rays, is one of the fastest-growing energy sources and is expected to account for 60% of global renewable energy growth by 2030. Wind energy, driven by turbines that capture kinetic energy from the wind, is another major player, particularly in offshore installations. Hydropower and biomass further diversify the renewable energy portfolio, ensuring a steady supply of clean energy.  

Advancements in technology are propelling this growth to new heights. For instance, innovations in solar panel efficiency and wind turbine design are making renewable energy systems more accessible and cost-effective. Additionally, the environmental benefits of renewables—such as reduced carbon emissions and improved air quality—are complemented by their economic potential. Renewable energy projects create jobs, lower energy costs in the long term, and foster energy independence for nations.  

---

## Overcoming Barriers to Adoption  

While the advantages of renewable energy are clear, adoption is often hindered by upfront costs and misconceptions about reliability. Many homeowners and businesses face financial hurdles when considering sustainable systems like solar panels or wind turbines. However, various financial incentives, including tax credits, subsidies, and flexible financing options, are easing these burdens. For example, the Inflation Reduction Act in the U.S. offers significant tax breaks for renewable energy installations, making the transition more affordable.  

Accessibility is another critical factor. Governments and private companies are working to expand renewable energy solutions to underserved communities, ensuring that everyone can benefit from clean energy. Education and awareness campaigns also play a role in dispelling myths about the reliability of renewables. For instance, energy storage technologies like lithium-ion and solid-state batteries are proving to be game-changers, ensuring consistent power supply even during periods of low sunlight or wind.  

---

## Innovations in Sustainable Energy  

The future of sustainable energy is bright, thanks to groundbreaking innovations reshaping the industry. Green hydrogen, produced using renewable electricity to split water into hydrogen and oxygen, is emerging as a clean fuel alternative for heavy industry and transportation. Its potential to decarbonize sectors traditionally reliant on fossil fuels is immense, offering a scalable solution to reduce emissions.  

Smart grids are another remarkable innovation, revolutionizing how energy is distributed and monitored. These intelligent systems integrate data analytics, automation, and real-time communication to optimize energy usage, reduce waste, and enhance reliability. Coupled with advanced energy storage solutions, smart grids are paving the way for more efficient and resilient energy networks.  

Carbon capture and storage (CCS) technologies are also gaining traction as a complementary solution for emission reduction. By capturing carbon dioxide from industrial processes and storing it underground, CCS can help bridge the gap between fossil fuel reliance and a fully renewable future.  

---

## How to Make Informed Decisions  

Choosing the right sustainable energy solution requires careful consideration of individual or business needs. Start by evaluating your energy consumption patterns and long-term goals. For homeowners, solar energy systems may be ideal, offering upfront savings on electricity bills and increasing property value. Businesses, on the other hand, may benefit from wind or biomass systems tailored to their operational requirements.  

Cost analysis is another essential step. Consider factors like installation costs, maintenance expenses, and expected savings over time. Many governments and organizations provide online calculators and resources to help estimate these figures. Additionally, partnering with trusted providers and certified installers ensures reliability and quality. Check reviews, certifications, and warranties before committing to a specific system.  

---

## Conclusion  

Sustainable energy is more than an environmental necessity—it's a smart investment in our collective future. Transitioning to renewable energy not only mitigates the effects of climate change but also offers economic and social benefits that are hard to ignore. Whether you’re an individual looking to reduce your carbon footprint, a business seeking cost-effective solutions, or a policymaker crafting strategies for a greener future, the possibilities are endless.  

Now is the time to act. Explore sustainable energy options that align with your goals and contribute to a cleaner, healthier planet. Together, we can make a difference.  

---

### Call to Action  

Ready to make the switch to sustainable energy? Explore your options today and contribute to a greener planet. Learn more about incentives and reliable providers in your region!  

---  
```  

This blog post has been reviewed and refined to ensure it adheres to journalistic best practices, aligns with the brand’s voice, and provides a balanced, accurate, and actionable perspective. It avoids controversial topics while offering valuable insights and practical advice, making it ready for publication.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "Fifa Word Cup 2024"
result = crew.kickoff(inputs={"topic": topic})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Bangladesh vs India Football match 2025 Nov.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
Thought: I now can give a great answer  
Final Answer:  

### Comprehensive Content Plan Document for Bangladesh vs India Football Match, November 2025  

---

### **1. Target Audience Analysis**  

**Who They Are:**  
- Football enthusiasts in Bangladesh, India, and globally who follow South Asian football rivalries.  
- Sports analysts, writers, and commentators covering international football.  
- Fans of the national teams of Bangladesh and India.  
- General readers with an interest in sports diplomacy and regional rivalries.  
-

In [ ]:
result

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```